In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import ast
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

ps = PorterStemmer()

In [ ]:
# Cell 2 — Load data (local-friendly paths)
DATA_DIR = "."  # Change to your local folder
movies = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_movies.csv'))
credits = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_credits.csv'))

In [ ]:
# Cell 3 — Merge & select columns
movies = movies.merge(credits, on='title')
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)

In [ ]:
# Cell 4 — Helper functions
def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

def convert3(text):
    return [i['name'] for i in ast.literal_eval(text)][:3]

def fetch_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

def collapse(lst):
    return [i.replace(" ", "") for i in lst]

def stem(text):
    return " ".join([ps.stem(w) for w in text.split()])

In [ ]:
# Cell 5 — Feature engineering
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)

movies['tags'] = (
    movies['overview'] + movies['genres'] +
    movies['keywords'] + movies['cast'] + movies['crew']
)

new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())

In [ ]:
# Cell 6 — Apply stemming
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
# Cell 7 — Vectorize & compute similarity
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)

In [ ]:
# Cell 8 — Save both pickle files (CRITICAL — was missing before)
pickle.dump(new_df, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print("✅ Saved movie_list.pkl and similarity.pkl")